# 03 — Jira Exploration: Sprint Intelligence

**Objective**: use `src/services/sprint_metrics.py` and `src/services/jira_normalizer.py` to build a portfolio-wide sprint intelligence view — completion trends, carryover determined by issue identity (not summary text), and a full data-quality report — and prove the identity-vs-text distinction on the real dataset.

**Dependencies**: `02_jira_connection.ipynb` (same client construction pattern).

**Configuration**: same `AS_OF = 2026-03-22` reference date as notebook 02, for comparable numbers.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date
import pandas as pd

from src.connectors.jira_client import build_default_jira_client
from src.services import jira_normalizer as norm
from src.services import sprint_metrics

client = build_default_jira_client()
AS_OF = date(2026, 3, 22)
PROJECT_KEYS = ["PHX", "ORCA", "NOVA", "TITAN", "LYNX", "QSR"]

## Portfolio sprint health

Most recent sprint per project, using `sprint_metrics.build_sprints` under the hood via `client.get_previous_sprints`.

In [ ]:
rows = []
for key in PROJECT_KEYS:
    latest = client.get_previous_sprints(key, count=1, as_of=AS_OF)
    if not latest:
        continue
    s = latest[0]
    blocked = client.get_blocked_issues(key, as_of=AS_OF)
    rows.append({
        "project": key,
        "latest_sprint": s.sprint_id,
        "committed_sp": s.committed_story_points,
        "completed_sp": s.completed_story_points,
        "completion_pct": round(s.completion_pct, 1) if s.completion_pct is not None else None,
        "carryover_sp": s.carryover_story_points,
        "blocked_issues": len(blocked.records),
    })

df = pd.DataFrame(rows)
df

,project,latest_sprint,committed_sp,completed_sp,completion_pct,carryover_sp,blocked_issues
0,PHX,PHX-SPR-3,35.0,28.0,80.0,7.0,4
1,ORCA,ORCA-SPR-3,36.0,14.0,38.9,22.0,8
2,NOVA,NOVA-SPR-3,63.0,30.0,47.6,33.0,10
3,TITAN,TITAN-SPR-5,69.0,20.0,29.0,49.0,8
4,LYNX,LYNX-SPR-5,67.0,27.0,40.3,40.0,8
5,QSR,QSR-SPR-1,14.0,9.0,64.3,5.0,0


These numbers match Phase 1's hand-computed sample report exactly (PHX 80.0%, ORCA 38.9%, NOVA 47.6%, TITAN 29.0%, LYNX 40.3%) — this time produced by the actual connector/services code instead of one-off pandas in a shell command.

## Carryover: identity vs. summary text

The mock data generator reuses template summaries across *different* tickets in *different* sprints. This is a real trap: a naive "did this summary appear last sprint too" check will misidentify unrelated tickets as one issue carrying over. `detect_carryover_issues` only ever compares `issue_key`; `naive_summary_carryover` is kept only as a labeled anti-pattern to prove the point.

In [ ]:
sprint1 = client.get_sprint_issues("PHX-SPR-1").records
sprint2 = client.get_sprint_issues("PHX-SPR-2").records

identity_based = sprint_metrics.detect_carryover_issues(sprint1, sprint2)
naive_text_based = sprint_metrics.naive_summary_carryover(sprint1, sprint2)

print(f"Identity-based carryover (correct):        {identity_based}")
print(f"Naive summary-based 'carryover' (WRONG):   {naive_text_based}")

Identity-based carryover (correct):        []
Naive summary-based 'carryover' (WRONG):   [('PHX-5', 'PHX-23')]


In [ ]:
# Show exactly why the naive matcher is wrong: pull up the two tickets it confused.
for key in ["PHX-5", "PHX-23"]:
    issue = next(i for i in sprint1 + sprint2 if i.issue_key == key)
    print(f"{issue.issue_key:8s} sprint={issue.sprint_id:12s} status={issue.status:10s} summary={issue.summary!r}")

PHX-5    sprint=PHX-SPR-1    status=DONE       summary='Resolve intermittent failure in API gateway'
PHX-23   sprint=PHX-SPR-2    status=TODO       summary='Resolve intermittent failure in API gateway'


`PHX-5` and `PHX-23` are two unrelated tickets in two different sprints that happen to share a summary. A text-matching "carryover detector" would report one issue moving forward across sprints; in reality these are unrelated work items and nothing carried over. `detect_carryover_issues` correctly returns `[]` for this pair because it only ever looks at `issue_key`.

This dataset's flat, single-snapshot structure means no `issue_key` is ever assigned to two different `sprint_id`s in the same pull — a genuinely-moved issue would need either (a) Jira's real changelog (`get_issue_history`, once a live source exists), or (b) two separate historical pulls of the same still-open issue. Both are out of scope for this connector-only phase; `detect_carryover_issues` is written generically so it becomes meaningful the moment either input is available, without changing its logic.

## Data quality report

Missing fields and unmapped status/blocker values, surfaced rather than silently patched, across the whole portfolio.

In [5]:
field_map = norm.load_field_mapping()
status_cfg = norm.load_status_mapping()

raw_rows, _meta = client._fetch_all_pages()
normalized = norm.normalize_all(raw_rows, field_map, status_cfg)

print(f"Total issues normalized: {len(normalized.issues)}")
print(f"Rows skipped (no issue_key): {normalized.report.rows_skipped}")
print(f"Unmapped raw statuses seen: {normalized.report.unmapped_statuses}")
print(f"Unmapped blocker values seen: {normalized.report.unmapped_blocker_values}")
print(f"Issues with a missing field flagged: {len(normalized.report.missing_field_reports)}")

Total issues normalized: 270
Rows skipped (no issue_key): []
Unmapped raw statuses seen: {}
Unmapped blocker values seen: {}
Issues with a missing field flagged: 42


In [6]:
missing_df = pd.DataFrame(
    [{"issue_key": r.issue_key, "missing_fields": ", ".join(r.missing_fields)} for r in normalized.report.missing_field_reports]
)
print(f"{len(missing_df)} issues have a flagged missing field (assignee/priority always checked; "
      f"story_points checked except for Epic/Sub-task, where it's structurally optional)\n")
missing_df.head(10)

42 issues have a flagged missing field (assignee/priority always checked; story_points checked except for Epic/Sub-task, where it's structurally optional)



,issue_key,missing_fields
0,PHX-5,story_points
1,PHX-13,story_points
2,PHX-21,story_points
3,ORCA-3,"assignee, story_points"
4,ORCA-8,story_points
5,ORCA-12,assignee
6,ORCA-18,story_points
7,ORCA-28,story_points
8,ORCA-34,story_points
9,NOVA-1,assignee


Zero unmapped statuses/blocker values means `config/status_mapping.yaml` fully covers this dataset's vocabulary — if this portfolio onboarded a team using Jira statuses like "Peer Review" that aren't in the map yet, they'd show up here instead of silently defaulting to something misleading.

## Blocker age distribution across the portfolio

In [7]:
all_blocked = []
for key in PROJECT_KEYS:
    all_blocked.extend(client.get_blocked_issues(key, as_of=AS_OF).records)

blocked_df = pd.DataFrame(
    [{"issue_key": i.issue_key, "project_id": i.project_id, "age_days": i.blocker_age_days, "reason": i.blocker_reason} for i in all_blocked]
).sort_values("age_days", ascending=False)

print(f"Total blocked issues portfolio-wide: {len(blocked_df)}")
blocked_df.head(10)

Total blocked issues portfolio-wide: 38


,issue_key,project_id,age_days,reason
22,TITAN-3,10004,85,"Unresolved dependency: TITAN-1, TITAN-2"
12,NOVA-8,10003,80,Flagged blocked via blocker_status field (sour...
5,ORCA-2,10002,76,Flagged blocked via blocker_status field (sour...
14,NOVA-11,10003,75,Flagged blocked via blocker_status field (sour...
23,TITAN-4,10004,67,Flagged blocked via blocker_status field (sour...
2,PHX-17,10001,63,Flagged blocked via blocker_status field (sour...
6,ORCA-11,10002,63,Flagged blocked via blocker_status field (sour...
13,NOVA-10,10003,62,Flagged blocked via blocker_status field (sour...
7,ORCA-22,10002,60,Unresolved dependency: ORCA-9
4,ORCA-1,10002,59,Flagged blocked via blocker_status field (sour...


## Validation checks

- [x] Sprint completion numbers reproduce Phase 1's hand-computed sample report exactly
- [x] Identity-based carryover returns `[]` for PHX Sprint 1 -> Sprint 2; the naive text matcher demonstrably returns the false positive `(PHX-5, PHX-23)`
- [x] Data-quality report runs across the full 270-issue portfolio with zero unmapped statuses/blocker values (config is complete for this dataset) and a bounded, inspectable list of missing-field issues
- [x] Blocker ages sort correctly and match the per-project spot checks in notebook 02

## Testing

`tests/test_sprint_metrics.py` (17 tests) covers `build_sprint` math, `derive_sprint_status` boundaries, blocker-age calculation, and — most importantly for this notebook — a regression test pinned to the real `PHX-5`/`PHX-23` collision so this exact trap can't silently start passing again if someone "simplifies" carryover detection back to summary matching later.

## Next step

Phase 3/4/5 territory: a live `JiraCloudRESTSource`, the financial connector, and the cross-source `Project` unification that joins this Jira-only view to budget data via `config/project_mapping.yaml`. Stopping here per the current task scope — Jira ingestion and sprint intelligence validated.